In [ ]:
# --- Cell 1: Cài đặt và Khai báo ---
# (Giữ nguyên)

from google.colab import drive
drive.mount('/content/drive')

!pip install -q tensorflow tensorflowjs matplotlib

import os, tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

tf.get_logger().setLevel('ERROR')

BASE_DIR = '/content/drive/MyDrive'
DATASET_PATH = f'{BASE_DIR}/dataset'
MODEL_PATH   = f'{BASE_DIR}/plant_model.keras' # Định dạng Keras hiện đại
TFJS_PATH    = f'{BASE_DIR}/plant_model_js'

os.makedirs(TFJS_PATH, exist_ok=True)
print(f"Các đường dẫn đã được thiết lập. Sẵn sàng cho Cell 2.")

In [ ]:
# --- Cell 2: Tải và Chuẩn bị Dữ liệu ---
# (Giữ nguyên)

IMG_SIZE, BATCH, EPOCHS_NEW, EPOCHS_FINE = 224, 32, 10, 5

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.2, horizontal_flip=True
)

train_gen = datagen.flow_from_directory(DATASET_PATH, target_size=(IMG_SIZE, IMG_SIZE),
                                        batch_size=BATCH, class_mode='categorical',
                                        subset='training', shuffle=True)
val_gen = datagen.flow_from_directory(DATASET_PATH, target_size=(IMG_SIZE, IMG_SIZE),
                                      batch_size=BATCH, class_mode='categorical',
                                      subset='validation', shuffle=False)

num_classes = len(train_gen.class_indices)
print(f"Đã tìm thấy {num_classes} lớp.")
print(f"Sẵn sàng cho Cell 3.")

In [9]:
# --- Cell 3: Xây dựng và Huấn luyện Model (ĐÃ CẢI TIẾN) ---

def build_model(num_classes):
    """Xây dựng model mới để huấn luyện lần đầu."""
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False  # Đóng băng toàn bộ base model
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(num_classes, activation='softmax')(x)
    model = Model(base.input, out)

    # Biên dịch với learning rate mặc định cho lần huấn luyện đầu
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

if os.path.exists(MODEL_PATH):
    print(f"✅ Đã tìm thấy model tại {MODEL_PATH}. Đang tải và cấu hình để fine-tuning...")
    model = load_model(MODEL_PATH)

    # Tìm base model (MobileNetV2) để unfreeze
    base_model_layer = None
    for layer in model.layers:
        if "mobilenet" in layer.name: # Tìm base model bằng tên
            base_model_layer = layer
            break

    if base_model_layer:
        base_model_layer.trainable = True
        print(f"Base model '{base_model_layer.name}' đã được đặt trainable = True.")

        # Tùy chọn: Unfreeze 20 lớp cuối của *toàn bộ* model
        for layer in model.layers[-20:]:
            if not isinstance(layer, Dropout):
                layer.trainable = True
    else:
        print("⚠️ Không tìm thấy lớp base model 'mobilenet', unfreeze các lớp cuối theo cách cũ.")
        for layer in model.layers[-20:]:
            if not isinstance(layer, Dropout):
                layer.trainable = True

    # *** CẢI TIẾN QUAN TRỌNG: Biên dịch lại model với learning rate RẤT NHỎ ***
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # 0.00001
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    print("✅ Model đã được biên dịch lại với learning rate 1e-5 cho fine-tuning.")
    epochs = EPOCHS_FINE

else:
    print("⛔ Không tìm thấy model. Đang xây dựng model mới...")
    model = build_model(num_classes)
    # Hàm build_model đã tự biên dịch với 'adam' (learning rate mặc định)
    epochs = EPOCHS_NEW

# Callbacks
callbacks = [
    ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

# Huấn luyện model
print(f"\nBắt đầu huấn luyện cho {epochs} epochs...")
history = model.fit(train_gen,
                    validation_data=val_gen,
                    epochs=epochs,
                    callbacks=callbacks)

print(f"Hoàn tất huấn luyện. Model tốt nhất đã được lưu tại {MODEL_PATH}.")
print(f"Sẵn sàng cho Cell 4.")

✅ Đã tìm thấy model tại /content/drive/MyDrive/plant_model.keras. Đang tải và cấu hình để fine-tuning...
⚠️ Không tìm thấy lớp base model 'mobilenet', unfreeze các lớp cuối theo cách cũ.
✅ Model đã được biên dịch lại với learning rate 1e-5 cho fine-tuning.

Bắt đầu huấn luyện cho 5 epochs...
Epoch 1/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.9963 - loss: 0.0112
Epoch 1: val_accuracy improved from -inf to 0.99113, saving model to /content/drive/MyDrive/plant_model.keras
240/240 ━━━━━━━━━━━━━━━━━━━━ 2516s 10s/step - accuracy: 0.9963 - loss: 0.0112 - val_accuracy: 0.9911 - val_loss: 0.0465
Epoch 2/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9966 - loss: 0.0135
Epoch 2: val_accuracy improved from 0.99113 to 0.99426, saving model to /content/drive/MyDrive/plant_model.keras
240/240 ━━━━━━━━━━━━━━━━━━━━ 627s 3s/step - accuracy: 0.9966 - loss: 0.0135 - val_accuracy: 0.9943 - val_loss: 0.0220
Epoch 3/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9985 - loss: 0

In [11]:
# --- Cell 4: Chuyển đổi sang TF.js (ĐÃ CẢI TIẾN) ---
# (Sử dụng định dạng SavedModel thay vì .h5)

# Cài đặt thư viện (nếu cell này chạy độc lập)
!pip install -q tensorflowjs h5py

import tensorflowjs as tfjs
import os
import tensorflow as tf
import shutil # Thư viện để xóa thư mục

# --- 1. KIỂM TRA BIẾN 'model' ---
try:
    _ = model.name
    print("✅ Đã tìm thấy 'model' trong bộ nhớ.")
except NameError:
    print("⛔ LỖI: Không tìm thấy biến 'model'.")
    # Nếu chạy độc lập, thử tải từ file keras đã lưu
    if os.path.exists(MODEL_PATH):
        print(f"Đang tải model từ {MODEL_PATH}...")
        model = load_model(MODEL_PATH)
    else:
        print("Vui lòng chạy lại Cell 3 (train model) trước khi chạy cell này.")
        raise

# --- 2. ĐỊNH NGHĨA ĐƯỜNG DẪN ---
# (Các biến đã có từ Cell 1)
# MODEL_PATH = f'{BASE_DIR}/plant_model.keras'
# TFJS_PATH  = f'{BASE_DIR}/plant_model_js'

# Đường dẫn cho thư mục SavedModel TẠM THỜI
SAVED_MODEL_TEMP_PATH = f'{BASE_DIR}/plant_model_saved_model_temp'

# --- 3. LƯU SANG ĐỊNH DẠNG SAVEDMODEL (Ưu tiên) ---
print(f"Đang chuẩn bị thư mục tạm SavedModel: {SAVED_MODEL_TEMP_PATH}")
# Xóa thư mục tạm cũ nếu có
if os.path.exists(SAVED_MODEL_TEMP_PATH):
    shutil.rmtree(SAVED_MODEL_TEMP_PATH)
os.makedirs(SAVED_MODEL_TEMP_PATH)

print(f"Đang lưu model (từ bộ nhớ) sang định dạng SavedModel tại: {SAVED_MODEL_TEMP_PATH}")
# Sử dụng model.export() để lưu ở định dạng SavedModel
model.export(SAVED_MODEL_TEMP_PATH)

# --- 4. LƯU FILE .KERAS (ĐỂ DÙNG SAU NÀY) ---
# Đảm bảo file .keras cũng được cập nhật phiên bản mới nhất
print(f"Đang lưu model (từ bộ nhớ) sang định dạng .keras tại: {MODEL_PATH}")
model.save(MODEL_PATH) # Lưu file .keras chính thức

# --- 5. CHUẨN BỊ THƯ MỤC OUTPUT TFJS ---
print(f"Đang chuẩn bị thư mục output TFJS: {TFJS_PATH}")
!rm -rf {TFJS_PATH}
os.makedirs(TFJS_PATH, exist_ok=True)

# --- 6. CHẠY CONVERTER TRÊN THƯ MỤC SAVEDMODEL ---
print("Đang chạy tensorflowjs_converter...")
# Chú ý: input_format bây giờ là 'tf_saved_model'
!tensorflowjs_converter --input_format=tf_saved_model \
                       {SAVED_MODEL_TEMP_PATH} \
                       {TFJS_PATH}

# --- 7. DỌN DẸP THƯ MỤC TẠM ---
print(f"Đang dọn dẹp thư mục tạm: {SAVED_MODEL_TEMP_PATH}")
shutil.rmtree(SAVED_MODEL_TEMP_PATH)
print("✅ Đã dọn dẹp thư mục tạm.")

# --- 8. GHI FILE LABELS ---
print("Đang ghi file labels.js...")
try:
    # Thử dùng biến train_gen từ Cell 2
    with open(os.path.join(TFJS_PATH, 'labels.js'), 'w') as f:
        f.write(f"const CLASS_NAMES = {list(train_gen.class_indices.keys())};")
    print("✅ Đã ghi labels.js từ 'train_gen'.")
except NameError:
    # Nếu chạy cell này độc lập, train_gen không tồn tại
    print("⚠️ Không tìm thấy biến 'train_gen', đang dùng danh sách dự phòng.")
    # Bạn nên cập nhật danh sách này nếu dataset thay đổi
    class_names_fallback = [
      "Apple___Apple_scab",
      "Apple___Black_rot",
      "Apple___Cedar_apple_rust",
      "Apple___healthy",
      "Blueberry___healthy",
    ]
    with open(os.path.join(TFJS_PATH, 'labels.js'), 'w') as f:
        f.write(f"const CLASS_NAMES = {class_names_fallback};")

print("\n--- HOÀN TẤT ---\n")
print(f"✅ Đã tạo file Keras: {MODEL_PATH}")
print(f"✅ Đã tạo thư mục TFJS: {TFJS_PATH} (từ SavedModel)")
print("\n🎉 Vui lòng tải thư mục 'plant_model_js' mới lên GitHub và kiểm tra lại trang web.")

✅ Đã tìm thấy 'model' trong bộ nhớ.
Đang chuẩn bị thư mục tạm SavedModel: /content/drive/MyDrive/plant_model_saved_model_temp
Đang lưu model (từ bộ nhớ) sang định dạng SavedModel tại: /content/drive/MyDrive/plant_model_saved_model_temp
Saved artifact at '/content/drive/MyDrive/plant_model_saved_model_temp'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer')]
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  134585121893328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134585096045392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134585096046736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134587574587984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134585096048080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134585096048272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134585096048848: T